# 简历数据采集 Notebook

每个 cell 单独跑，把输出数字填到简历里。

## Cell 1: 环境准备

In [ ]:
import sys, time
sys.path.insert(0, "/mnt/data0/jupyter/RAG")
import config
from inference_client import InferenceClient
from vector_store import QdrantSearchClient

inf = InferenceClient()
qd = QdrantSearchClient()
print("✅ 环境就绪")

## Cell 2: 单次检索平均耗时

连续测 10 次取平均。

In [ ]:
query = "电梯故障报修没人处理"
times = []
for _ in range(10):
    start = time.perf_counter()
    vec = inf.get_embedding(query)
    results = qd.search(query_vector=vec, top_k=5)
    times.append((time.perf_counter() - start) * 1000)

avg = sum(times) / len(times)
print(f"min={min(times):.0f}ms  max={max(times):.0f}ms  avg={avg:.0f}ms")
print(f"\n📝 简历: 单次语义检索平均耗时约 {avg:.0f}ms")

## Cell 3: 链路耗时拆解

把向量化和数据库查询分开计时。

In [ ]:
query = "楼上漏水导致天花板损坏"
vec_times, qd_times = [], []

for _ in range(5):
    t0 = time.perf_counter()
    vec = inf.get_embedding(query)
    t1 = time.perf_counter()
    qd.search(query_vector=vec, top_k=5)
    t2 = time.perf_counter()
    vec_times.append((t1-t0)*1000)
    qd_times.append((t2-t1)*1000)

v_avg = sum(vec_times)/len(vec_times)
q_avg = sum(qd_times)/len(qd_times)
print(f"向量化: {v_avg:.0f}ms  |  数据库查询: {q_avg:.0f}ms  |  总计: {v_avg+q_avg:.0f}ms")
print(f"\n📝 简历: 向量化约 {v_avg:.0f}ms，数据库查询约 {q_avg:.0f}ms")

## Cell 4: 并发效率（4线程 vs 串行）

In [ ]:
from embedding_worker import EmbeddingWorkerPool

inf2 = InferenceClient()
qd2 = QdrantSearchClient()

queries = ["电梯故障","楼上漏水","垃圾清理","停车收费",
           "物业费高","楼道灯坏","噪音投诉","门禁损坏"]

# 4线程并发
pool = EmbeddingWorkerPool(inf2, qd2)
pool.start()
futs = [pool.submit(q) for q in queries]
t0 = time.perf_counter()
_ = [f.result(timeout=60) for f in futs]
pt = (time.perf_counter()-t0)*1000
pool.shutdown()

# 串行
t0 = time.perf_counter()
for q in queries:
    qd2.search(query_vector=inf2.get_embedding(q), top_k=5)
st = (time.perf_counter()-t0)*1000

speedup = st/pt if pt > 0 else 0
print(f"串行: {st:.0f}ms  |  4线程并发: {pt:.0f}ms  |  加速比: {speedup:.1f}x")
print(f"\n📝 简历: 4线程并发，效率相比串行提升约 {speedup:.0f} 倍")
inf2.close(); qd2.close()

## Cell 5: 数据治理指标（不需要跑，直接抄）

In [ ]:
print("""
📝 直接填简历的数据治理指标:

  总样本: 8882条 (训练6216 / 验证1775 / 测试891)
  标签层级: 15类一级标签 + 三层标签体系
  有效标签覆盖率: 88.5%→95.6% (+8%)
  修复标签映射错位: 7类
  恢复被丢弃样本: 427条
  统一两份结构迥异的Excel → 标准JSONL Schema
""")

## Cell 6: 技术栈清单（不需要跑，直接抄）

In [ ]:
print("""
📝 技能栏关键词:

  Python | Pandas | FastAPI | Qdrant | Qwen3
  OpenAI v1 API | 向量检索 | RAG | 多线程并发
  Jupyter Notebook | Git | 数据治理
""")

In [ ]:
inf.close()
qd.close()
print("✅ 全部完成")